# 07 — NCCS vs IRS Form 990: 2023 Source Comparison

**Goal:** Determine whether NCCS `efile_v3_1` can reproduce our existing 2023 Form 990 filing identities, self-reported significant-diversion labels, and core financial/governance fields **before** expanding to earlier years.

**Scope:** Data-source feasibility only. No model training, feature selection, or 2025 holdout analysis. Do not treat a disclosure as proven fraud or missing values as negative labels.

**Run one section at a time.** The notebook stops before comparing data if the NCCS files have not been downloaded. Review column inventories before configuring mappings; do not assume that similarly named fields are equivalent.

Source: https://nccs.urban.org/nccs/catalogs/catalog-efile-v3_1.html

2023 NCCS tables to download as **Parquet**, full Form 990 (not `-EZ`):
- `F9-P00-T00-HEADER-2023`
- `F9-P01-T00-SUMMARY-2023`
- `F9-P06-T00-GOVERNANCE-2023`

Place all three files in `data/external/nccs/efile_v3_1/2023/`. Use the catalog's Parquet links. **Do not download 15 years yet.**


## 1. Setup and paths

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / ".gitignore").exists()), None)
if ROOT is None:
    ROOT = here.parent if here.name == "notebooks" else here

NCCS_DIR = ROOT / "data/external/nccs/efile_v3_1/2023"
NCCS_DIR.mkdir(parents=True, exist_ok=True)
IRS_PATH = ROOT / "data/analytics/modeling/05_form990_modeling_organization_year_v1.parquet"
REPORT_DIR = ROOT / "reports/nccs_comparison_2023"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TABLES = {
    "header": "F9-P00-T00-HEADER-2023",
    "summary": "F9-P01-T00-SUMMARY-2023",
    "governance": "F9-P06-T00-GOVERNANCE-2023",
}
print("Project root:", ROOT)
print("NCCS folder:", NCCS_DIR)
print("IRS model file exists:", IRS_PATH.exists())


## 2. Confirm downloaded NCCS files and inspect schemas

This intentionally uses filename matching, because NCCS download filenames may include extensions or release prefixes. Missing files produce actionable instructions rather than a silent fallback.

In [ ]:
if not IRS_PATH.exists():
    raise FileNotFoundError(f"Existing IRS modeling file not found: {IRS_PATH}")
if not NCCS_DIR.exists():
    raise FileNotFoundError(f"Create {NCCS_DIR} and download the three 2023 Parquet tables from the NCCS catalog.")

nccs_paths = {}
for name, stem in TABLES.items():
    matches = sorted(p for p in NCCS_DIR.glob("*.parquet") if stem.lower() in p.name.lower())
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one Parquet file matching {stem} in {NCCS_DIR}; found {len(matches)}. Files: {[p.name for p in matches]}")
    nccs_paths[name] = matches[0]

inventory = []
for name, path in nccs_paths.items():
    pf = pq.ParquetFile(path)
    inventory.append({"table": name, "file": path.name, "rows": pf.metadata.num_rows, "columns": len(pf.schema_arrow.names)})
display(pd.DataFrame(inventory))
for name, path in nccs_paths.items():
    cols = pq.ParquetFile(path).schema_arrow.names
    print(f"\n{name.upper()}: {len(cols)} columns")
    display(pd.DataFrame({"column": cols}).head(100))


## 3. Load existing IRS 2023 records and NCCS tables

The IRS file is already **canonicalized to one organization–tax-year**; NCCS tables are **one row per filing**. Therefore, comparing raw counts directly is not a like-for-like comparison.

In [ ]:
irs = pd.read_parquet(IRS_PATH)
irs = irs.loc[pd.to_numeric(irs["tax_year"], errors="coerce").eq(2023)].copy()
nccs = {name: pd.read_parquet(path) for name, path in nccs_paths.items()}
print("IRS 2023 canonical organization-years:", len(irs))
print("IRS columns:", len(irs.columns))
print("NCCS row counts:", {name: len(df) for name, df in nccs.items()})
print("IRS identifiers and candidate comparison fields:")
display(pd.DataFrame({"column": [c for c in irs.columns if any(t in c.lower() for t in ["ein", "filing", "tax_year", "label", "revenue", "asset", "expense", "board", "independent"])]}))

## 4. Discover candidate NCCS identifiers, target, and financial columns

Inspect these candidates and their values before making any assumptions about their semantics. In particular, do not automatically equate a missing checkbox with `No`.

In [ ]:
patterns = {
    "identity": r"ein|objectid|tax.?year|return.?type|form.?type|tax.?period|amend",
    "diversion": r"diversion|asset.divert|asset_div",
    "financial": r"revenue|expense|asset|liabil|net.asset",
    "board": r"indep|voting|govern|director|trustee|family|relation",
}
for table, df in nccs.items():
    print("\n", table.upper())
    for group, pattern in patterns.items():
        found = [c for c in df.columns if re.search(pattern, str(c), re.I)]
        print(f"{group}: {found[:45]}{' ...' if len(found)>45 else ''}")

In [ ]:
# NCCS filing coverage and duplicate analysis

def summarize_nccs_table(df, name):
    temp = df.copy()

    temp["ein_clean"] = (
        temp["ORG_EIN"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(9)
    )

    temp["tax_year_clean"] = pd.to_numeric(
        temp["TAX_YEAR"], errors="coerce"
    )

    return {
        "table": name,
        "total_rows": len(temp),
        "unique_eins": temp["ein_clean"].nunique(),
        "unique_objectids": temp["OBJECTID"].nunique(),
        "unique_ein_years": temp[
            ["ein_clean", "tax_year_clean"]
        ].drop_duplicates().shape[0],
        "rows_tax_year_2023": (
            temp["tax_year_clean"] == 2023
        ).sum(),
        "duplicate_ein_year_rows": temp.duplicated(
            ["ein_clean", "tax_year_clean"]
        ).sum(),
        "amended_returns": (
            temp["RETURN_AMENDED_X"]
            .astype("string")
            .str.lower()
            .isin(["true", "1", "yes", "x"])
        ).sum()
    }


coverage = pd.DataFrame([
    summarize_nccs_table(nccs["header"], "Header"),
    summarize_nccs_table(nccs["summary"], "Summary"),
    summarize_nccs_table(nccs["governance"], "Governance")
])

display(coverage.T)

In [ ]:
# Compare 2023 IRS and NCCS coverage and diversion labels

irs_compare = irs.copy()

irs_compare["ein_clean"] = (
    irs_compare["ein"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(9)
)

irs_compare["tax_year_clean"] = pd.to_numeric(
    irs_compare["tax_year"], errors="coerce"
)

irs_compare = irs_compare[
    irs_compare["tax_year_clean"] == 2023
].copy()

# Start with the NCCS Governance table because it contains our target.
nccs_compare = nccs["governance"].copy()

nccs_compare["ein_clean"] = (
    nccs_compare["ORG_EIN"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(9)
)

nccs_compare["tax_year_clean"] = pd.to_numeric(
    nccs_compare["TAX_YEAR"], errors="coerce"
)

nccs_compare = nccs_compare[
    nccs_compare["tax_year_clean"] == 2023
].copy()

# Examine raw label values before converting them to 0/1.
print("IRS label values:")
display(irs_compare["fraud_label"].value_counts(dropna=False))

print("NCCS diversion disclosure values:")
display(
    nccs_compare["F9_06_GVRN_ASSET_DIVERSION_X"]
    .value_counts(dropna=False)
)

# Compare unique EIN coverage without selecting among amended filings.
irs_eins = set(irs_compare["ein_clean"].dropna())
nccs_eins = set(nccs_compare["ein_clean"].dropna())

coverage_comparison = pd.DataFrame({
    "metric": [
        "IRS unique EINs",
        "NCCS unique EINs",
        "EINs in both",
        "IRS only",
        "NCCS only"
    ],
    "count": [
        len(irs_eins),
        len(nccs_eins),
        len(irs_eins & nccs_eins),
        len(irs_eins - nccs_eins),
        len(nccs_eins - irs_eins)
    ]
})

display(coverage_comparison)

In [ ]:
# Compare IRS and NCCS diversion labels for matched EINs

label_col = "F9_06_GVRN_ASSET_DIVERSION_X"

nccs_labels = nccs_compare[
    ["ein_clean", "OBJECTID", "RETURN_AMENDED_X", label_col]
].copy()

# Normalize mixed boolean, numeric, and string representations.
normalized = (
    nccs_labels[label_col]
    .astype("string")
    .str.strip()
    .str.lower()
)

nccs_labels["label_normalized"] = normalized.map({
    "true": 1,
    "1": 1,
    "false": 0,
    "0": 0
}).astype("Int64")

# Aggregate all filings for each EIN.
nccs_by_ein = (
    nccs_labels
    .groupby("ein_clean", as_index=False)
    .agg(
        nccs_positive_any=("label_normalized", lambda s: int(s.eq(1).any())),
        nccs_negative_any=("label_normalized", lambda s: int(s.eq(0).any())),
        nccs_nonmissing_count=("label_normalized", "count"),
        nccs_filing_count=("OBJECTID", "nunique")
    )
)

# Preserve unknowns rather than treating missing as negative.
nccs_by_ein["nccs_label"] = pd.Series(
    pd.NA,
    index=nccs_by_ein.index,
    dtype="Int64"
)

has_observation = nccs_by_ein["nccs_nonmissing_count"] > 0

nccs_by_ein.loc[has_observation, "nccs_label"] = (
    nccs_by_ein.loc[has_observation, "nccs_positive_any"]
    .astype("Int64")
)

# Match against the IRS canonical organization-year records.
comparison = irs_compare[
    ["ein_clean", "fraud_label"]
].merge(
    nccs_by_ein,
    on="ein_clean",
    how="inner",
    validate="one_to_one"
)

comparison["irs_label"] = pd.to_numeric(
    comparison["fraud_label"], errors="coerce"
).astype("Int64")

print("Matched organizations:", len(comparison))

print("\nIRS vs NCCS labels (including missing):")
display(
    pd.crosstab(
        comparison["irs_label"].fillna(-1),
        comparison["nccs_label"].fillna(-1),
        rownames=["IRS label"],
        colnames=["NCCS label (-1 = missing)"],
        margins=True
    )
)

print("\nNCCS filing conflicts:")
display(
    comparison[
        ["nccs_positive_any", "nccs_negative_any"]
    ].value_counts(dropna=False)
)

print("\nPositive agreement:")
display(pd.DataFrame([{
    "irs_positive": int(comparison["irs_label"].eq(1).sum()),
    "nccs_positive": int(comparison["nccs_label"].eq(1).sum()),
    "positive_in_both": int(
        (
            comparison["irs_label"].eq(1) &
            comparison["nccs_label"].eq(1)
        ).sum()
    ),
    "irs_positive_nccs_not_positive": int(
        (
            comparison["irs_label"].eq(1) &
            comparison["nccs_label"].ne(1).fillna(True)
        ).sum()
    ),
    "nccs_positive_irs_not_positive": int(
        (
            comparison["nccs_label"].eq(1) &
            comparison["irs_label"].ne(1).fillna(True)
        ).sum()
    )
}]))

In [ ]:
# Identify comparable financial and board fields

nccs_fields = {
    "revenue_cy": "F9_01_REV_TOT_CY",
    "revenue_py": "F9_01_REV_TOT_PY",
    "expenses_cy": "F9_01_EXP_TOT_CY",
    "expenses_py": "F9_01_EXP_TOT_PY",
    "assets_boy": "F9_01_NAFB_ASSET_TOT_BOY",
    "assets_eoy": "F9_01_NAFB_ASSET_TOT_EOY",
    "liabilities_eoy": "F9_01_NAFB_LIAB_TOT_EOY",
    "net_assets_eoy": "F9_01_NAFB_TOT_EOY",
    "employees_count": "F9_01_ACT_GVRN_EMPL_TOT",
    "volunteers_count": "F9_01_ACT_GVRN_VOL_TOT",
    "voting_board_members": "F9_01_ACT_GVRN_NUM_VOTE_MEMB",
    "independent_board_members": "F9_01_ACT_GVRN_NUM_VOTE_MEMB_IND"
}

irs_columns = set(irs_compare.columns)
nccs_columns = set(nccs["summary"].columns)

field_mapping = pd.DataFrame([
    {
        "feature": feature,
        "irs_column": feature if feature in irs_columns else None,
        "nccs_column": nccs_col,
        "irs_available": feature in irs_columns,
        "nccs_available": nccs_col in nccs_columns
    }
    for feature, nccs_col in nccs_fields.items()
])

display(field_mapping)

print("\nIRS columns potentially related to financial or board features:")

keywords = [
    "revenue",
    "expense",
    "asset",
    "liabil",
    "employee",
    "volunteer",
    "board",
    "voting"
]

matching_irs_columns = [
    col for col in irs_compare.columns
    if any(word in col.lower() for word in keywords)
]

print("\n".join(sorted(matching_irs_columns)))

In [ ]:
# Compare IRS and NCCS financial values
# Restrict NCCS to EINs with exactly one Summary filing.

summary_compare = nccs["summary"].copy()

summary_compare["ein_clean"] = (
    summary_compare["ORG_EIN"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(9)
)

summary_compare["tax_year_clean"] = pd.to_numeric(
    summary_compare["TAX_YEAR"], errors="coerce"
)

summary_compare = summary_compare[
    summary_compare["tax_year_clean"] == 2023
].copy()

# Keep only EINs with one filing to avoid amended-return ambiguity.
filing_counts = summary_compare.groupby("ein_clean")["OBJECTID"].transform("nunique")

summary_single = summary_compare.loc[
    filing_counts == 1
].copy()

print("NCCS single-filing organizations:", len(summary_single))

# Select and rename NCCS columns.
nccs_financial = summary_single[
    ["ein_clean"] + list(nccs_fields.values())
].rename(
    columns={v: f"nccs_{k}" for k, v in nccs_fields.items()}
)

# Select corresponding IRS columns.
irs_financial = irs_compare[
    ["ein_clean"] + list(nccs_fields.keys())
].rename(
    columns={k: f"irs_{k}" for k in nccs_fields.keys()}
)

# One-to-one EIN match.
financial_comparison = irs_financial.merge(
    nccs_financial,
    on="ein_clean",
    how="inner",
    validate="one_to_one"
)

print("Matched organizations for financial comparison:", len(financial_comparison))

# Compare numeric values and preserve missingness.
results = []

for feature in nccs_fields:
    irs_values = pd.to_numeric(
        financial_comparison[f"irs_{feature}"],
        errors="coerce"
    )

    nccs_values = pd.to_numeric(
        financial_comparison[f"nccs_{feature}"],
        errors="coerce"
    )

    both_present = irs_values.notna() & nccs_values.notna()
    both_missing = irs_values.isna() & nccs_values.isna()

    equal_values = pd.Series(
        False,
        index=financial_comparison.index
    )

    equal_values.loc[both_present] = np.isclose(
        irs_values.loc[both_present],
        nccs_values.loc[both_present],
        rtol=0,
        atol=0
    )

    results.append({
        "feature": feature,
        "matched_records": len(financial_comparison),
        "both_present": int(both_present.sum()),
        "exact_matches": int(equal_values.sum()),
        "value_mismatches": int((both_present & ~equal_values).sum()),
        "both_missing": int(both_missing.sum()),
        "irs_missing_only": int(
            (irs_values.isna() & nccs_values.notna()).sum()
        ),
        "nccs_missing_only": int(
            (irs_values.notna() & nccs_values.isna()).sum()
        ),
        "match_rate_when_both_present": (
            float(equal_values.sum() / both_present.sum())
            if both_present.sum() else np.nan
        )
    })

financial_results = pd.DataFrame(results)

display(
    financial_results.sort_values(
        "match_rate_when_both_present"
    )
)

In [ ]:
# Investigate financial mismatches

mismatch_summary = []
mismatch_examples = []

for feature in nccs_fields:
    irs_values = pd.to_numeric(
        financial_comparison[f"irs_{feature}"],
        errors="coerce"
    )

    nccs_values = pd.to_numeric(
        financial_comparison[f"nccs_{feature}"],
        errors="coerce"
    )

    both_present = irs_values.notna() & nccs_values.notna()

    mismatched = pd.Series(
        False,
        index=financial_comparison.index
    )

    mismatched.loc[both_present] = ~np.isclose(
        irs_values.loc[both_present],
        nccs_values.loc[both_present],
        rtol=0,
        atol=0
    )

    differences = (
        irs_values.loc[mismatched] -
        nccs_values.loc[mismatched]
    ).abs()

    mismatch_summary.append({
        "feature": feature,
        "mismatch_count": int(mismatched.sum()),
        "median_absolute_difference": (
            differences.median() if len(differences) else np.nan
        ),
        "max_absolute_difference": (
            differences.max() if len(differences) else np.nan
        )
    })

    if mismatched.any():
        examples = financial_comparison.loc[
            mismatched,
            [
                "ein_clean",
                f"irs_{feature}",
                f"nccs_{feature}"
            ]
        ].head(5).copy()

        examples["feature"] = feature

        examples = examples.rename(columns={
            f"irs_{feature}": "irs_value",
            f"nccs_{feature}": "nccs_value"
        })

        mismatch_examples.append(examples)

display(
    pd.DataFrame(mismatch_summary)
    .sort_values("mismatch_count", ascending=False)
)

print("\nExample mismatches:")

if mismatch_examples:
    display(
        pd.concat(
            mismatch_examples,
            ignore_index=True
        )
    )

## 5. Set reviewed column mappings

**Manual review gate.** Fill these values from the displayed schema and NCCS data dictionary. Do not proceed with guesses. `OBJECTID` (if present) is useful for a filing-level match, while EIN + tax year alone may represent multiple filings. Leave optional comparisons as `None` until confirmed.

In [ ]:
MAPPING = {
    "header_ein": "ORG_EIN",
    "header_tax_year": "TAX_YEAR",
    "header_filing_id": "OBJECTID",
    "governance_filing_id": "OBJECTID",
    "summary_filing_id": "OBJECTID",
    "governance_diversion": "F9_06_GVRN_ASSET_DIVERSION_X",
    "summary_revenue": "F9_01_REV_TOT_CY",
    "summary_expenses": "F9_01_EXP_TOT_CY",
    "summary_assets": "F9_01_NAFB_ASSET_TOT_EOY"
}

print("Mappings configured.")

## 6. Validate mappings and compare filing identity coverage

This is a **filing-level** comparison when both systems share a compatible filing identifier. If they do not, use EIN + tax year only for an initial coverage comparison; never claim an exact filing match on that basis.

In [ ]:
required = ["header_ein", "header_tax_year", "header_filing_id", "governance_filing_id", "summary_filing_id", "governance_diversion"]
missing = [key for key in required if not MAPPING[key]]
if missing:
    raise ValueError(f"Fill required MAPPING entries before comparison: {missing}")
for key, table in [("header_ein", "header"), ("header_tax_year", "header"), ("header_filing_id", "header"), ("governance_filing_id", "governance"), ("summary_filing_id", "summary"), ("governance_diversion", "governance"), ("summary_revenue", "summary"), ("summary_expenses", "summary"), ("summary_assets", "summary")]:
    value = MAPPING[key]
    if value and value not in nccs[table].columns:
        raise KeyError(f"{key}={value!r} not in {table} columns")

def norm_ein(s):
    return s.astype("string").str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True).str.zfill(9)

def norm_id(s):
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

h = nccs["header"].copy()
h["comparison_ein"] = norm_ein(h[MAPPING["header_ein"]])
h["comparison_year"] = pd.to_numeric(h[MAPPING["header_tax_year"]], errors="coerce")
h["comparison_id"] = norm_id(h[MAPPING["header_filing_id"]])
h = h.loc[h["comparison_year"].eq(2023)].copy()

irs["comparison_ein"] = norm_ein(irs["ein"])
irs["comparison_year"] = pd.to_numeric(irs["tax_year"], errors="coerce")
print("NCCS 2023 header filings:", len(h))
print("NCCS distinct EIN/year:", h[["comparison_ein", "comparison_year"]].drop_duplicates().shape[0])
print("IRS canonical 2023 EIN/year:", irs[["comparison_ein", "comparison_year"]].drop_duplicates().shape[0])
print("NCCS duplicate filing IDs:", h["comparison_id"].duplicated().sum())
print("IRS duplicate EIN/year:", irs.duplicated(["comparison_ein", "comparison_year"]).sum())

coverage = irs[["comparison_ein", "comparison_year"]].drop_duplicates().merge(
    h[["comparison_ein", "comparison_year"]].drop_duplicates(),
    on=["comparison_ein", "comparison_year"], how="outer", indicator=True
)
print("EIN/year coverage (not filing-level equivalence):")
display(coverage["_merge"].value_counts().rename_axis("coverage").to_frame("organization_years"))


## 7. Join NCCS governance and summary on filing identifier

Stop if join keys are not unique. This protects against accidentally multiplying rows and distorting diversion rates.

In [ ]:
g = nccs["governance"].copy()
s = nccs["summary"].copy()
g["comparison_id"] = norm_id(g[MAPPING["governance_filing_id"]])
s["comparison_id"] = norm_id(s[MAPPING["summary_filing_id"]])
for name, frame in [("header", h), ("governance", g), ("summary", s)]:
    if frame["comparison_id"].isna().any() or frame["comparison_id"].duplicated().any():
        raise ValueError(f"{name} filing identifier is null or nonunique; investigate before joining")

selected_g = g[["comparison_id", MAPPING["governance_diversion"]]].rename(columns={MAPPING["governance_diversion"]: "nccs_diversion_raw"})
selected_s = [MAPPING[k] for k in ("summary_revenue", "summary_expenses", "summary_assets") if MAPPING[k]]
selected_s = list(dict.fromkeys(selected_s))
nccs_joined = h[["comparison_id", "comparison_ein", "comparison_year"]].merge(
    selected_g, on="comparison_id", how="left", validate="one_to_one", indicator="governance_match"
).merge(s[["comparison_id"] + selected_s], on="comparison_id", how="left", validate="one_to_one", indicator="summary_match")
print("Governance join:")
display(nccs_joined["governance_match"].value_counts())
print("Summary join:")
display(nccs_joined["summary_match"].value_counts())
print("Raw diversion responses (not yet recoded):")
display(nccs_joined["nccs_diversion_raw"].astype("string").value_counts(dropna=False).head(30))


## 8. Compare raw label distributions and overlapping organization-years

This section intentionally does **not** convert NCCS checkbox values to binary automatically. Determine the documented Yes/No encoding first. Multiple NCCS filings per EIN/year are retained as an ambiguity requiring a filing-selection rule.

In [ ]:
print("IRS 2023 existing label:")
display(irs["fraud_label"].value_counts(dropna=False))
print("NCCS raw diversion response:")
display(nccs_joined["nccs_diversion_raw"].astype("string").value_counts(dropna=False))

counts = nccs_joined.groupby(["comparison_ein", "comparison_year"], dropna=False).size().rename("nccs_filings_in_org_year").reset_index()
joined = irs[["comparison_ein", "comparison_year", "fraud_label"]].merge(
    counts, on=["comparison_ein", "comparison_year"], how="left", validate="one_to_one"
)
print("NCCS filing multiplicity for existing IRS organization-years:")
display(joined["nccs_filings_in_org_year"].fillna(0).value_counts().head(15))
print("IRS positive cases by NCCS match count:")
display(joined.groupby(joined["nccs_filings_in_org_year"].fillna(0))["fraud_label"].agg(["count", "sum"]).head(15))


### Conclusion: NCCS vs. IRS Data Validation

**Objective:** Determine whether the NCCS Form 990 e-file dataset is a reliable alternative to our existing IRS XML-derived data, with the goal of expanding our dataset to approximately 15 years.

**Key Findings:**
- **Organization coverage:** NCCS contains 99.95% of the organizations in our existing 2023 IRS dataset, plus approximately 206,000 additional organizations.
- **Target validation:** All 315 matched positive significant diversion of assets disclosures agree between datasets.
- **Financial and governance accuracy:** All 12 compared fields have greater than 99.98% agreement where both datasets contain values.
- **Table integrity:** Governance records join successfully to 100% of Header filings; Summary records join to 99.71%.
- **Duplicate filings:** Approximately 98% of matched organizations have a single NCCS filing. Multiple filings will require a consistent selection rule.
- **Data discrepancies:** A small number of financial differences were identified, but they are not widespread enough to prevent proceeding with NCCS.

**Conclusion:** NCCS demonstrates strong consistency with our existing IRS data and is suitable for further historical integration and modeling evaluation. Historical data quality and label completeness still need to be validated.

**Next Steps:** Use NCCS as the primary source to investigate 2010–2024 coverage, construct a standardized organization-year dataset, engineer longitudinal financial and governance features, and retrain the significant diversion disclosure prediction models.